In [16]:
"""
Lab Assignment 3 - Topic Modelling

Name: Muhammad Faris Irfan Bin Mohmad Natar, Faiq Arifbillah Bin Syahrir 
Student ID: IS01083410, IS01083886

Coherence Score Analysis:
The coherence score of 0.6527 indicates that the LDA model has successfully identified meaningful and logically consistent topics within the news dataset. In topic modeling, a score in the 0.60–0.70 range is generally considered good to excellent, suggesting that the top words in each topic are closely related and will be easily interpretable by a human reader. This result confirms that the preprocessing steps (lemmatization and stopword removal) were effective, allowing the model to capture distinct themes rather than just statistical noise.
"""

'\nLab Assignment 3 - Topic Modelling\n\nName: Muhammad Faris Irfan Bin Mohmad Natar, Faiq Arifbillah Bin Syahrir \nStudent ID: IS01083410, IS01083886\n\nCoherence Score Analysis:\n'

In [6]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel

# For Coherence Score
from gensim.models import CoherenceModel

import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\faris\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\faris\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\faris\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [7]:
df = pd.read_csv('news_dataset.csv')
documents = df['text'].tolist()

In [8]:
stop_words = set(stopwords.words('english')) # Create a set of English stopwords
lemmatizer = WordNetLemmatizer() # Initialize a WordNet lemmatizer
def preprocess_text(text):
    # Check if text is null or not a string
    if not isinstance(text, str):
        return [] 
    tokens = word_tokenize(text.lower())
    tokens = [token for token in tokens if token.isalnum()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens
preprocessed_documents = [preprocess_text(doc) for doc in documents] # Preprocess each document in the list
print(preprocessed_documents[0])

['wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'sport', 'car', 'looked', 'late', 'early', '70', 'called', 'bricklin', 'door', 'really', 'small', 'addition', 'front', 'bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please']


In [9]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)
# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)
# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

In [10]:
# Run LDA
lda_model = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15) # Train an LDA model on the corpus with 5 topics using Gensim's LdaModel class

In [11]:
# empty list to store dominant topic labels for each document
article_labels = []
# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
 # for each document, convert to bag-of-words representation
 bow = dictionary.doc2bow(doc)
 # get list of topic probabilities
 topics = lda_model.get_document_topics(bow)
 # determine topic with highest probability
 dominant_topic = max(topics, key=lambda x: x[1])[0]
 # append to the list
 article_labels.append(dominant_topic)
# Create DataFrame
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})
# Print the DataFrame
print("Table with Articles and Topic:")
print(df_result)
print()

Table with Articles and Topic:
                                                 Article  Topic
0      I was wondering if anyone out there could enli...      4
1      I recently posted an article asking what kind ...      4
2      \nIt depends on your priorities.  A lot of peo...      4
3      an excellent automatic can be found in the sub...      4
4      : Ford and his automobile.  I need information...      4
...                                                  ...    ...
11309  Secrecy in Clipper Chip\n\nThe serial number o...      3
11310  Hi !\n\nI am interested in the source of FEAL ...      3
11311  The actual algorithm is classified, however, t...      3
11312  \n\tThis appears to be generic calling upon th...      2
11313  \nProbably keep quiet and take it, lest they g...      4

[11314 rows x 2 columns]



In [12]:
# Print top terms for each topic
for topic_id in range(lda_model.num_topics):
 print(f"Top terms for Topic #{topic_id}:")
 top_terms = lda_model.show_topic(topic_id, topn=10)
 print([term[0] for term in top_terms])
 print()

Top terms for Topic #0:
['1', '0', 'q', 'max', '2', '7', 'g', 'r', '3', 'p']

Top terms for Topic #1:
['year', 'db', 'team', 'game', 'new', 'president', 'national', 'university', 'american', 'first']

Top terms for Topic #2:
['would', 'people', 'one', 'think', 'say', 'know', 'right', 'time', 'u', 'god']

Top terms for Topic #3:
['x', 'key', 'file', 'use', 'system', 'chip', 'window', 'encryption', 'program', 'information']

Top terms for Topic #4:
['would', 'one', 'get', 'like', 'know', 'good', 'drive', 'time', 'problem', 'could']



In [13]:
# Print the top terms for each topic with weight
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
 print(f"Topic {idx}:")
 terms = [term.strip() for term in topic.split("+")]
 for term in terms:
     weight, word = term.split("*")
 print(f"- {word.strip()} (weight: {weight.strip()})")
 print()


Top Terms for Each Topic:
Topic 0:
- "p" (weight: 0.025)

Topic 1:
- "first" (weight: 0.004)

Topic 2:
- "god" (weight: 0.005)

Topic 3:
- "information" (weight: 0.006)

Topic 4:
- "could" (weight: 0.005)



In [14]:
# texts=preprocessed_documents must be a list of lists of strings
coherence_model_lda = CoherenceModel(model=lda_model, 
                                     texts=preprocessed_documents, 
                                     dictionary=dictionary, 
                                     coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print(f'\nCoherence Score: {coherence_lda:.4f}')


Coherence Score: 0.6527
